In [ ]:
# --- repo bootstrap (auto-added) ---
# Run paths relative to the repo root and make src/ importable.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
_SRC = os.path.abspath("src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%reload_ext autoreload

In [3]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [4]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

MGSD Test set balance:
 label
unrelated     250
stereotype    250
Name: count, dtype: int64
MGSD Few-shot examples balance:
 label
stereotype    5
unrelated     5
Name: count, dtype: int64
MentalManip Test set balance:
 manipulative
1    250
0    250
Name: count, dtype: int64
MentalManip Few-shot examples balance:
 manipulative
1    5
0    5
Name: count, dtype: int64


In [ ]:
from dotenv import load_dotenv

import os
import openai
from openai import OpenAI
from anthropic import Anthropic
from mistralai import Mistral
import cohere
import google.generativeai as genai
from xai_sdk import Client as XAIClient


load_dotenv()


ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_DEEPSEEK": "DeepSeek",
    "API_KEY_GROK": "Grok",
    "API_KEY_ANTHROPIC": "Anthropic",
    "API_KEY_GEMINI": "Gemini",
    "API_KEY_MISTRAL": "Mistral",
    "API_KEY_COHERE": "Cohere",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        print(f"Warning: {name} - API key missing in .env file.")
        continue

API_KEY_OPENAI = os.getenv("API_KEY_OPENAI")
API_KEY_DEEPSEEK = os.getenv("API_KEY_DEEPSEEK")
API_KEY_ANTHROPIC = os.getenv("API_KEY_ANTHROPIC")
API_KEY_GEMINI = os.getenv("API_KEY_GEMINI")
API_KEY_MISTRAL = os.getenv("API_KEY_MISTRAL")
API_KEY_COHERE = os.getenv("API_KEY_COHERE")
API_KEY_GROK = os.getenv("API_KEY_GROK")


backend_to_run = [
    #"openai-4.1-mini",
    #"openai-4o-mini",
    #"mistral-small-2506",
    #"mistral-small-2503",
    #"anthropic-sonnet",
    #"deepseek-v3-chat",
    #"gemini-2.5-flash",
]

backends = {
    "openai-4.1-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4.1-mini-2025-04-14",
        "fname":   "openai_4.1_mini"
    },
    "openai-4o-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4o-mini-2024-07-18",
        "fname":   "openai_4o_mini"
    },

    "deepseek-v3-chat": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_DEEPSEEK, base_url="https://api.deepseek.com"),
        "model":   "deepseek-chat",
        "fname":   "deepseek_v3"
    },

    "anthropic-sonnet": {
        "provider": "anthropic",
        "client":  Anthropic(api_key=API_KEY_ANTHROPIC),
        "model":   "claude-3-7-sonnet-latest",
        "fname":   "anthropic_3_7_sonnet"
    },

    "gemini-2.5-flash": {
        "provider": "gemini",
        "client":  (genai.configure(api_key=API_KEY_GEMINI) or genai.GenerativeModel("gemini-2.5-flash")),
        "model":   "gemini-2.5-flash",
        "fname":   "google_gemini_2_5_flash"
    },

    "mistral-small-2506": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2506",
        "fname":   "mistral_small_2506"
    },
    "mistral-small-2503": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2503",
        "fname":   "mistral_small_2503"
    },
}


In [12]:
from tqdm import tqdm
import os, json
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

from tree_of_thought_v2 import TreeOfThoughtExplorer
from tree_of_thought_judge import PathSelectionJudge

from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case


from profiles.schema import PersonSet
from profiles.profile_dict import PERSON_SEEDS

for backend in backend_to_run:

    model = backends[backend]["model"]
    client = backends[backend]["client"]
    model_filename = backends[backend]["fname"]

    case_name_set = ["stereotype", "manipulation"]      # "manipulation"
    max_branching_factor = 3
    max_depth = 2
    n_shots = 1
    max_tokens_dict = {"generation": 500}
    
    
    role_playing_mode = "none"
    selected_profiles = [None]          #  ["profile1","profile2"] for role_playing_mode != "none"
    person_set = PersonSet(seeds=PERSON_SEEDS, metadata={})
    
    
    
    base_dir = f"results/{model_filename}/tot_explorer"
    os.makedirs(base_dir, exist_ok=True)
    
    try:
        for profile_n in selected_profiles:
            for case_name in case_name_set:
                print(f"\n=== Running ToT Explorer + Judge for {case_name} ===")
    
                if case_name.lower() == "manipulation":
                    case = manipulation_case
                    task_definition = manipulation_definition_short
                    data = sample_mentalmanip.iloc[:400]
                
                elif case_name.lower() == "stereotype":
                    case = stereotypes_case
                    task_definition = stereotype_definition_short_binary
                    data = sample_mgsd.iloc[:400]
                else:
                    raise ValueError(f"Unknown case name: {case_name}")
    
                if role_playing_mode == "none" or profile_n is None:
                    out_dir = f"{base_dir}/baseline"
                else:
                    out_dir = f"{base_dir}/role_playing/{profile_n}_{role_playing_mode}"
                os.makedirs(out_dir, exist_ok=True)
    
                output_file = f"{out_dir}/results_{case_name.lower()}_tot_explorer.csv"
                reasoning_file = f"{out_dir}/reasoning_{case_name.lower()}_tot_explorer.json"
    
                explorer = TreeOfThoughtExplorer(
                    case=case,
                    client=client,
                    model=model,
                    max_branching_factor=max_branching_factor,
                    max_depth=max_depth,
                    task_definition=task_definition,
                    max_tokens_dict=max_tokens_dict,
                    n_shots=n_shots,
                    examples_df=None,
                )
    
                judge = PathSelectionJudge(
                    client=client,
                    model=model,
                    temperature=0.0,
                    max_tokens=256,
                    person_key=profile_n if role_playing_mode != "none" else None,
                    role_playing=role_playing_mode,
                    person_set=person_set
                )
    
                rows = []
                detailed = []
    
                for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
                    text = row[case.input_col]
                    true_label = row[case.label_col]
                    if isinstance(true_label, str):
                        true_label = true_label.strip()
    
                    try:
                        paths = explorer.solve(text)
                        selection = judge.choose_best_from_explorer(case=case, explorer_paths=paths, max_paths=12)
    
                        selected_path_id = selection.get("path_id")
                        selected_label = selection.get("label")
    
                        if not selected_label:
                            if paths and paths[0] and paths[0][-1].verdict:
                                selected_label = paths[0][-1].verdict
                            else:
                                selected_label = list(case.valid_labels)[-1]
    
                        mapped_label = case.label_map.get(
                            str(selected_label).strip(), list(case.label_map.values())[-1]
                        )
    
                        rows.append({
                            "sample_id": idx,
                            "text": text,
                            "true_label": true_label,
                            "pred_label": mapped_label,
                            "raw_pred_label": selected_label,
                            "selected_path_id": selected_path_id,
                            "role_playing": role_playing_mode,
                            "person_key": profile_n if role_playing_mode != "none" else None,
                        })
    
    
                        all_paths_serialized = []
                        for p in paths:
                            all_paths_serialized.append({
                                "path_id": "->".join(t.id for t in p),
                                "steps": [
                                    {
                                        "id": t.id,
                                        "content": t.content,
                                        "label": t.verdict
                                    } for t in p
                                ]
                            })
    
                        detailed.append({
                            "sample_id": idx,
                            "judge_raw": selection.get("raw"),
                            "selected_path_id": selected_path_id,
                            "selected_label": selected_label,
                            "paths": all_paths_serialized,
                        })
    
                    except Exception as e:
                        print(f"Error processing sample {idx}: {e}")
                        continue
    
                if not rows:
                    print(f"No successful selections for {case_name}")
                    continue
    
                df_out = pd.DataFrame(rows)
                os.makedirs(os.path.dirname(output_file), exist_ok=True)
                df_out.to_csv(output_file, index=False)
                print(f"=== Saved {len(df_out)} rows to {output_file} ===")
    
                os.makedirs(os.path.dirname(reasoning_file), exist_ok=True)
                with open(reasoning_file, "w", encoding="utf-8") as f:
                    json.dump(detailed, f, indent=2, ensure_ascii=False)
                print(f"=== Saved detailed reasoning to {reasoning_file} ===")
    
                try:
                    if case_name.lower() == "manipulation":
                        y_true = df_out["true_label"].astype(int)
                        y_pred = df_out["pred_label"].astype(int)
                    elif case_name.lower() == "stereotype":
                        y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                        y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()
    
                    print(f"\n=== Classification Report for {case_name} (ToT Explorer + Judge) ===")
                    print(classification_report(y_true, y_pred, zero_division=0))
                    print(f"\n=== Confusion Matrix for {case_name} ===")
                    labels = sorted(set(y_true) | set(y_pred))
                    print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))
    
                    accuracy = (y_true == y_pred).mean()
                    print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")
    
                    print(f"\n=== Label Distribution ===")
                    print("True labels:")
                    print(pd.Series(y_true).value_counts())
                    print("Predicted labels:")
                    print(pd.Series(y_pred).value_counts())
    
                except Exception as e:
                    print(f"Error in evaluation for {case_name}: {e}")
    
                print(f"\n=== Sample Chosen Paths (raw/parsed) ===")
                for i, item in enumerate(detailed[:3]):
                    print(f"\nSample {item['sample_id']}:")
                    tr = df_out.loc[df_out["sample_id"] == item["sample_id"], "true_label"].values
                    print(f"True Label: {tr[0] if len(tr) else 'N/A'}")
                    print(f"Selected Label: {item['selected_label']}")
                    print(f"Selected Path ID: {item['selected_path_id']}")
                    chosen = next((p for p in item["paths"] if p["path_id"] == item["selected_path_id"]), None)
                    if chosen:
                        print("Chosen Path Steps:")
                        for step in chosen["steps"]:
                            print(f"  {step['id']}: {step['content'][:180]} (Label: {step.get('label')})")
                    else:
                        print("Chosen path not found in serialized paths.")
                    print("Judge Raw (truncated):")
                    jr = item.get("judge_raw") or ""
                    print((jr[:300] + "...") if len(jr) > 300 else jr)
                    print("-" * 50)
    
    except Exception as e:
        print(f"== Error: {e} ==")
        import traceback
        traceback.print_exc()
    
    print("\n=== ToT Explorer + Judge Evaluation Complete ===")



=== Running ToT Explorer + Judge for stereotype ===


Processing stereotype:   4%|▍         | 15/400 [01:37<37:51,  5.90s/it]

[ERROR] Failed to parse JSON from model output:
```json
[
  { "thought": "The statement expresses a desire to visit Russia and assess the frustration levels of Russians, which implies a generalized assumption about the behavior or emotions of a large group of people based on their nationality." },
  { "thought": "The phrase 'how frustrating Russi
→ Expecting value: line 1 column 1 (char 0)


Processing stereotype:   9%|▉         | 35/400 [04:16<46:32,  7.65s/it]

[ERROR] Failed to parse JSON from model output:
```json
[
  { "thought": "The phrase 'very pear' does not reference any specific social group or identity marker, making it impossible to associate it with a stereotype.", "label": "No" },
  { "thought": "The statement explicitly denies that the phrase is a stereotype or generalization, which aligns
→ Expecting value: line 1 column 1 (char 0)


...

=== Saved 400 rows to results/mistral_small_2503/tot_explorer/baseline/results_stereotype_tot_explorer.csv ===
=== Saved detailed reasoning to results/mistral_small_2503/tot_explorer/baseline/reasoning_stereotype_tot_explorer.json ===

=== Classification Report for stereotype (ToT Explorer + Judge) ===
              precision    recall  f1-score   support

  stereotype       0.64      0.58      0.61       205
   unrelated       0.60      0.67      0.63       195

    accuracy                           0.62       400
   macro avg       0.62      0.62      0.62       400
weighted avg       0.62      0.62      0.62       400


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         118         87
unrelated           65        130

=== Accuracy for stereotype: 62.00% ===

=== Label Distribution ===
True labels:
true_label
stereotype    205
unrelated     195
Name: count, dtype: int64
Predicted labels:
pred_label
unrelated     217
stereotype    183
Name: 

Processing manipulation:  72%|███████▏  | 288/400 [23:57<08:02,  4.31s/it]

[ERROR] Failed to parse JSON from model output:
[
  { "thought": "Person1 uses a series of statements that imply Person2's actions reveal their character, which could be seen as a form of emotional pressure to conform to Person1's expectations." },
  { "thought": "Person1 frames the conversation in a way that suggests Person2 has no choice but to
→ Unterminated string starting at: line 4 column 16 (char 444)


Processing manipulation: 100%|██████████| 400/400 [33:35<00:00,  5.04s/it]

=== Saved 400 rows to results/mistral_small_2503/tot_explorer/baseline/results_manipulation_tot_explorer.csv ===
=== Saved detailed reasoning to results/mistral_small_2503/tot_explorer/baseline/reasoning_manipulation_tot_explorer.json ===

=== Classification Report for manipulation (ToT Explorer + Judge) ===
              precision    recall  f1-score   support

           0       0.67      0.27      0.38       197
           1       0.55      0.87      0.68       203

    accuracy                           0.57       400
   macro avg       0.61      0.57      0.53       400
weighted avg       0.61      0.57      0.53       400


=== Confusion Matrix for manipulation ===
    0    1
0  53  144
1  26  177

=== Accuracy for manipulation: 57.50% ===

=== Label Distribution ===
True labels:
true_label
1    203
0    197
Name: count, dtype: int64
Predicted labels:
pred_label
1    321
0     79
Name: count, dtype: int64

=== Sample Chosen Paths (raw/parsed) ===

Sample 0:
True Label: 1
Selected